# CS 228 experiment (Kaggle)

**Settings (right panel, Session options):** Accelerator = **GPU T4 x2** (or P100), **Internet = On**.
Run the cells **one at a time** (click the cell, Cmd+Enter) and read each output before the next.
The dataset is downloaded by cell 2, so you do NOT need to "Add Input".

In [ ]:
# 1. Get the code
%cd /kaggle/working
!rm -rf cs228-pad-project
GITHUB_USER = "ShivamShelar101"
!git clone https://github.com/{GITHUB_USER}/cs228-pad-project.git
%cd cs228-pad-project
!pip install -q onnxscript matplotlib

In [ ]:
# 2. Find (or download) the dataset. The first download is about 4 GB and takes a few minutes.
import os
SRC = "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing"
if not os.path.exists(SRC):
    import kagglehub
    SRC = kagglehub.dataset_download("attentionlayer241/celeba-spoof-for-face-antispoofing")
print("dataset at:", SRC)

In [ ]:
# 3. Build the face-crop dataset. This copy is partial, so the script uses only images that exist.
!python -m data.prepare_celeba_spoof --src {SRC} --dst ./data_pad --per_class 3000 --test_per_class 600
!for s in train val test; do for c in live spoof; do echo $s/$c $(ls data_pad/$s/$c | wc -l); done; done

Read the six counts above. If a count is under about 300, stop and ask for help before training.

In [ ]:
# 4. Sanity check: one quick epoch of the baseline
!python -m training.train_baseline --data_root ./data_pad --epochs 1 --seed 99 --out checkpoints/sanity

`val_acc` should be well above 0.5 (say 0.8+). If it is stuck near 0.5, stop: the data or labels are wrong.

In [ ]:
# 5. The experiment: baseline + 3 hardened variants + evaluation + report.
#    Pilot = EPOCHS 5, one seed. Keep the tab open, or use Save Version -> Save & Run All.
EPOCHS = 5
SEEDS = "0"
!EPOCHS={EPOCHS} SEEDS="{SEEDS}" bash scripts/run_experiment.sh ./data_pad

In [ ]:
# 6. Results table and chart
print(open("results/report.md").read())
from IPython.display import Image
Image("results/report.png")

In [ ]:
# 7. Attacked images for the live demo (PNG; only kept if they still fool the baseline)
LAST = EPOCHS - 1
!python -m scripts.make_attacked_images --data_root ./data_pad \
    --baseline checkpoints/baseline/seed0_best.pt \
    --hardened checkpoints/full/seed0_best.pt --n 8 --out demo_images

In [ ]:
# 8. Pack everything into bundle.zip, then download it from the Output panel
!EPOCHS={EPOCHS} bash scripts/bundle_outputs.sh